In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [4]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')

---

In [5]:
promotion.head()

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,M,33,2017-04-21,72000.0,8.57,1,1,1,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,M,33,2017-04-21,72000.0,14.11,1,1,1,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,M,33,2017-04-21,72000.0,10.27,1,0,1,0,0
3,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,informational,0,0,4,...,M,33,2017-04-21,72000.0,NaN,1,1,0,0,0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,informational,0,0,3,...,M,33,2017-04-21,72000.0,NaN,1,1,0,0,0


In [6]:
# bogo, discount만 필터링
bd_df = promotion[promotion['offer_type'].isin(['bogo', 'discount'])].copy()

In [7]:
channels = ['web', 'email', 'mobile', 'social']

# 채널 조합 전체
channel_funnel_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    channel_funnel_list.append({
        'channel': channel,
        'total_offers': len(temp),
        'viewed': temp['is_viewed'].sum(),
        'completed': temp['is_completed'].sum(),
        'view_rate(%)': round(temp['is_viewed'].sum() / len(temp) * 100, 2),
        #'view_drop_rate(%)': round((1 - temp['is_viewed'].sum() / len(temp)) * 100, 2),
        'completion_rate(%)': round(temp['is_completed'].sum() / temp['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - temp['is_completed'].sum() / temp['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': temp['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(temp['is_normal_flow'].sum() / len(temp) * 100, 2)
    })

channel_funnel = pd.DataFrame(channel_funnel_list)
display(channel_funnel)

# 채널 x offer_type
channel_type_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    for offer_type, group in temp.groupby('offer_type'):
        channel_type_list.append({
            'channel': channel,
            'offer_type': offer_type,
            'total_offers': len(group),
            'viewed': group['is_viewed'].sum(),
            'completed': group['is_completed'].sum(),
            'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
            #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
            'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
            #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
            'normal_flow_cnt': group['is_normal_flow'].sum(),
            'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
        })

channel_type_funnel = pd.DataFrame(channel_type_list)
display(channel_type_funnel)

# 채널 x offer_id
channel_id_list = []
for channel in channels:
    temp = bd_df[bd_df[channel] == 1].copy()
    for offer_id, group in temp.groupby('offer_id'):
        channel_id_list.append({
            'channel': channel,
            'offer_id': offer_id,
            'total_offers': len(group),
            'viewed': group['is_viewed'].sum(),
            'completed': group['is_completed'].sum(),
            'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
            #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
            'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
            #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
            'normal_flow_cnt': group['is_normal_flow'].sum(),
            'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
        })

channel_id_funnel = pd.DataFrame(channel_id_list)
display(channel_id_funnel)

,channel,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,53384,40178,29466,75.26,73.34,20685,38.75
1,email,61042,46894,33101,76.82,70.59,23267,38.12
2,mobile,53374,44231,29788,82.87,67.35,21967,41.16
3,social,38065,35942,21530,94.42,59.90,17759,46.65


,channel,offer_type,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,bogo,22841,18733,11866,82.01,63.34,8359,36.60
1,web,discount,30543,21445,17600,70.21,82.07,12326,40.36
2,email,bogo,30499,25449,15501,83.44,60.91,10941,35.87
3,email,discount,30543,21445,17600,70.21,82.07,12326,40.36
4,mobile,bogo,30499,25449,15501,83.44,60.91,10941,35.87
5,mobile,discount,22875,18782,14287,82.11,76.07,11026,48.20
6,social,bogo,22822,21278,11198,93.23,52.63,8835,38.71
7,social,discount,15243,14664,10332,96.20,70.46,8924,58.54


,channel,offer_id,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
0,web,bogo_10_10_5,7593,7298,3301,96.11,45.23,2739,36.07
1,web,bogo_5_5_5,7571,7264,4262,95.95,58.67,3514,46.41
2,web,bogo_5_5_7,7677,4171,4303,54.33,103.16,2106,27.43
3,web,discount_10_2_10,7597,7327,5230,96.45,71.38,4576,60.23
4,web,discount_10_2_7,7632,4118,3955,53.96,96.04,2102,27.54
5,web,discount_20_5_10,7668,2663,3313,34.73,124.41,1300,16.95
6,web,discount_7_3_7,7646,7337,5102,95.96,69.54,4348,56.87
7,email,bogo_10_10_5,7593,7298,3301,96.11,45.23,2739,36.07
8,email,bogo_10_10_7,7658,6716,3635,87.70,54.12,2582,33.72
9,email,bogo_5_5_5,7571,7264,4262,95.95,58.67,3514,46.41


In [8]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)
display(bd_df['channel_combo'].value_counts())

channel_combo
web+email+mobile+social    30407
web+email+mobile           15309
web+email                   7668
email+mobile+social         7658
Name: count, dtype: int64

In [9]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)

# 채널 조합 전체
combo_funnel_list = []
for combo, group in bd_df.groupby('channel_combo'):
    combo_funnel_list.append({
        'channel_combo': combo,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_funnel = pd.DataFrame(combo_funnel_list).sort_values('total_offers', ascending=False)
display(combo_funnel)

# 채널 조합 x offer_type
combo_type_list = []
for (combo, offer_type), group in bd_df.groupby(['channel_combo', 'offer_type']):
    combo_type_list.append({
        'channel_combo': combo,
        'offer_type': offer_type,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_type_funnel = pd.DataFrame(combo_type_list).sort_values('total_offers', ascending=False)
display(combo_type_funnel)

# 채널 조합 x offer_id
combo_id_list = []
for (combo, offer_id), group in bd_df.groupby(['channel_combo', 'offer_id']):
    combo_id_list.append({
        'channel_combo': combo,
        'offer_id': offer_id,
        'total_offers': len(group),
        'viewed': group['is_viewed'].sum(),
        'completed': group['is_completed'].sum(),
        'view_rate(%)': round(group['is_viewed'].sum() / len(group) * 100, 2),
        #'view_drop_rate(%)': round((1 - group['is_viewed'].sum() / len(group)) * 100, 2),
        'completion_rate(%)': round(group['is_completed'].sum() / group['is_viewed'].sum() * 100, 2),
        #'completion_drop_rate(%)': round((1 - group['is_completed'].sum() / group['is_viewed'].sum()) * 100, 2),
        'normal_flow_cnt': group['is_normal_flow'].sum(),
        'normal_flow_rate(%)': round(group['is_normal_flow'].sum() / len(group) * 100, 2)
    })

combo_id_funnel = pd.DataFrame(combo_id_list).sort_values('total_offers', ascending=False)
display(combo_id_funnel)

,channel_combo,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
3,web+email+mobile+social,30407,29226,17895,96.12,61.23,15177,49.91
2,web+email+mobile,15309,8289,8258,54.14,99.63,4208,27.49
1,web+email,7668,2663,3313,34.73,124.41,1300,16.95
0,email+mobile+social,7658,6716,3635,87.70,54.12,2582,33.72


,channel_combo,offer_type,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
5,web+email+mobile+social,discount,15243,14664,10332,96.20,70.46,8924,58.54
4,web+email+mobile+social,bogo,15164,14562,7563,96.03,51.94,6253,41.24
2,web+email+mobile,bogo,7677,4171,4303,54.33,103.16,2106,27.43
1,web+email,discount,7668,2663,3313,34.73,124.41,1300,16.95
0,email+mobile+social,bogo,7658,6716,3635,87.70,54.12,2582,33.72
3,web+email+mobile,discount,7632,4118,3955,53.96,96.04,2102,27.54


,channel_combo,offer_id,total_offers,viewed,completed,view_rate(%),completion_rate(%),normal_flow_cnt,normal_flow_rate(%)
2,web+email+mobile,bogo_5_5_7,7677,4171,4303,54.33,103.16,2106,27.43
1,web+email,discount_20_5_10,7668,2663,3313,34.73,124.41,1300,16.95
0,email+mobile+social,bogo_10_10_7,7658,6716,3635,87.70,54.12,2582,33.72
7,web+email+mobile+social,discount_7_3_7,7646,7337,5102,95.96,69.54,4348,56.87
3,web+email+mobile,discount_10_2_7,7632,4118,3955,53.96,96.04,2102,27.54
6,web+email+mobile+social,discount_10_2_10,7597,7327,5230,96.45,71.38,4576,60.23
4,web+email+mobile+social,bogo_10_10_5,7593,7298,3301,96.11,45.23,2739,36.07
5,web+email+mobile+social,bogo_5_5_5,7571,7264,4262,95.95,58.67,3514,46.41


(정상 흐름)